In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# operation fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. The cap is inherited by the fork-based run_parallel workers,
# which share the parent frame copy-on-write, so it stays compatible with them.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 10.8G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298144768

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from parallel_compute import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Feature-selection objectives and where each objective's outputs are written.
FEATURE_OBJECTIVES = ["normal", "arcsinh", "spikes", "dips"]
SELECTED_FEATURES_DIR = variables.CWD / "4_Features_select" / "Selected_features"

def build_label(objective, y_raw):
    # Label + LightGBM params per objective (matches how each model is trained later).
    y_raw = np.asarray(y_raw, dtype=np.float32)
    base = dict(learning_rate=0.1, num_leaves=31, min_child_samples=20, bagging_fraction=0.5,
                bagging_freq=1, force_col_wise=True, random_state=42, n_jobs=1, num_threads=1, verbose=-1)
    if objective == "normal":
        return y_raw, {**base, "objective": "regression_l1", "metric": "l1"}
    if objective == "arcsinh":
        return np.arcsinh(y_raw / variables.PRICE_TRANSFORM_SCALE).astype(np.float32), {**base, "objective": "regression", "metric": "rmse"}
    if objective == "spikes":
        return (y_raw > variables.SPIKE_THRESHOLD).astype(np.float32), {**base, "objective": "binary", "metric": "binary_logloss"}
    if objective == "dips":
        return (y_raw < variables.DIP_THRESHOLD).astype(np.float32), {**base, "objective": "binary", "metric": "binary_logloss"}
    raise ValueError(objective)


Inputs

In [3]:
import pyarrow.parquet as pq

FOR_SELECTION = variables.FEATURES_DATASET_FOR_SELECTION_PATH

# The window is ~5.5 GB; loading it whole AND keeping the 300k-row subsample copy
# is what exhausts RAM. Read only what's needed here - the feature names and the
# row index (cheap) - and align the targets. The next cell gathers just the
# sampled rows straight from disk, so the full window never lives in memory.
_pf = pq.ParquetFile(FOR_SELECTION)
_idx_cols = {c for c in (_pf.schema_arrow.pandas_metadata or {}).get("index_columns", []) if isinstance(c, str)}
feature_names = [n for n in _pf.schema_arrow.names if n not in _idx_cols]
feature_index = pd.read_parquet(FOR_SELECTION, columns=[]).index

targets = pd.read_parquet(variables.AGG_TARGET_DATASET_PATH)
# Align targets to the feature rows by timestamp (features start later than targets).
targets = targets.reindex(feature_index)

print(f"{len(feature_index):,} rows x {len(feature_names):,} features; targets {targets.shape}")


525,673 rows x 2,851 features; targets (525673, 96)


Subset sample

In [4]:
from tqdm.auto import tqdm

# Gather the sampled rows to a DISK memmap - materialising a 3+ GB subsample in
# RAM is what OOM-kills the kernel. Peak RAM here is one parquet batch; the next
# cell bins this memmap straight off disk, so the full subsample never needs to
# fit in memory. (Selected_features is on disk, not tmpfs.)
seed = np.random.default_rng(42)
n_samples = min(variables.FEATURE_SELECTION_SUBSAMPLE_AMOUNT, len(feature_index))
index = np.sort(seed.choice(len(feature_index), size=n_samples, replace=False))

targets_subset_index = targets.iloc[index].to_numpy(dtype=np.float32, copy=True)

_SUBSAMPLE_MMAP = str(SELECTED_FEATURES_DIR / "_ranking_subsample.tmp.dat")
features_subset_mm = np.memmap(_SUBSAMPLE_MMAP, dtype=np.float32, mode="w+", shape=(n_samples, len(feature_names)))
pf = pq.ParquetFile(FOR_SELECTION, pre_buffer=False, memory_map=False)
row0 = 0
_n_batches = max(1, -(-len(feature_index) // 50_000))
for rb in tqdm(pf.iter_batches(batch_size=50_000, columns=feature_names, use_threads=False),
               total=_n_batches, desc="Gathering subsample", unit="batch", leave=False):
    n = rb.num_rows
    lo = np.searchsorted(index, row0)
    hi = np.searchsorted(index, row0 + n)
    if hi > lo:
        arr = rb.to_pandas().to_numpy(dtype=np.float32)
        features_subset_mm[lo:hi] = arr[index[lo:hi] - row0]
        del arr
    row0 += n
features_subset_mm.flush()
print("subset:", features_subset_mm.shape, "(on disk)")


/home/daniel-davaris/venv_main/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                                       

subset: (400000, 2851) (on disk)


Core logic

Rank features by LightGBM **gain importance**, independently for each objective
(normal / arcsinh / spikes / dips) and each horizon. Tasks = objectives x horizons.


In [5]:
import gc
from tqdm.auto import tqdm

# Bin the on-disk subsample in batches via lgb.Sequence, so the full 300k x F
# matrix is never held in RAM (that materialisation is what crashed the kernel).
# Every task ranks the SAME features - only the label differs - so bin ONCE and
# reuse the binned Dataset via set_label. Identical results, full subsample,
# peak RAM of one batch + the binned copy. Each train() still uses all cores.
_n_threads = max(1, (os.cpu_count() or 4) - 2)


class _MmapSequence(lgb.Sequence):
    def __init__(self, mm, batch_size=20_000):
        self.mm = mm
        self.batch_size = batch_size

    def __getitem__(self, idx):
        return np.asarray(self.mm[idx], dtype=np.float64)  # lgb.Sequence requires float64

    def __len__(self):
        return self.mm.shape[0]


# bin_construct_sample_cnt caps the rows LightGBM stacks (as float64) to compute
# bin edges; the 200k default alone is a 4+ GB spike at this width, so shrink it.
base_dataset = lgb.Dataset(
    _MmapSequence(features_subset_mm),
    params={"bin_construct_sample_cnt": 20_000},
    free_raw_data=True,
).construct()

TASKS = [(objective, horizon_index)
         for objective in FEATURE_OBJECTIVES
         for horizon_index in range(len(targets.columns))]

ranked_importances = []
for objective, horizon_index in tqdm(TASKS, desc="Ranking", unit="task"):
    label, params = build_label(objective, targets_subset_index[:, horizon_index])
    base_dataset.set_label(np.asarray(label, dtype=np.float64))
    booster = lgb.train({**params, "num_threads": _n_threads}, base_dataset, num_boost_round=200)
    # Positional gain importance aligns with feature_names (Dataset column order).
    gain = booster.feature_importance(importance_type="gain").astype(np.float32)
    ranked_importances.append((objective, horizon_index, gain))
    del booster
    gc.collect()

# Drop the on-disk subsample scratch file now ranking is done.
del base_dataset, features_subset_mm
gc.collect()
if os.path.exists(_SUBSAMPLE_MMAP):
    os.remove(_SUBSAMPLE_MMAP)



[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.


Ranking: 100%|██████████| 384/384 [13:07:26<00:00, 123.04s/task]  


Clean up

In [6]:
horizon_cols = list(targets.columns)

# Regroup the flat (objective, horizon, gain) results by objective.
gains_by_objective = {objective: {} for objective in FEATURE_OBJECTIVES}
for objective, horizon_index, gain in ranked_importances:
    gains_by_objective[objective][horizon_index] = gain


def build_ranked_ordered(gains_by_horizon):
    values = pd.DataFrame(
        {horizon_cols[hi]: gains_by_horizon[hi] for hi in range(len(horizon_cols))},
        index=feature_names,
    ).astype(np.float32)
    # Order rows by mean gain so the de-dup step keeps the strongest of each
    # correlated group (this global ordering is load-bearing downstream).
    values = values.loc[values.mean(axis=1).sort_values(ascending=False).index]
    # Per-horizon integer rank (1 = most important), the layout consumed downstream.
    ranked_ordered = values.rank(axis=0, ascending=False, method="min").astype(int)
    return values, ranked_ordered


ranked_ordered_by_objective = {}
for objective in FEATURE_OBJECTIVES:
    values, ranked_ordered = build_ranked_ordered(gains_by_objective[objective])
    ranked_ordered_by_objective[objective] = ranked_ordered
    out_path = SELECTED_FEATURES_DIR / f"FEATURES_RANKED_ORDERED_{objective}.parquet"
    values.to_csv(f"Selected_features/features_ranked_values_{objective}.csv")
    ranked_ordered.to_parquet(out_path)
    print(f"{objective}: {ranked_ordered.shape[0]} features x {ranked_ordered.shape[1]} horizons -> {out_path.name}")


normal: 2851 features x 96 horizons -> FEATURES_RANKED_ORDERED_normal.parquet
arcsinh: 2851 features x 96 horizons -> FEATURES_RANKED_ORDERED_arcsinh.parquet
spikes: 2851 features x 96 horizons -> FEATURES_RANKED_ORDERED_spikes.parquet
dips: 2851 features x 96 horizons -> FEATURES_RANKED_ORDERED_dips.parquet


View

In [7]:
for objective in FEATURE_OBJECTIVES:
    print(objective)
    display(ranked_ordered_by_objective[objective].iloc[:3, :5])


normal


,target_h1,target_h2,target_h3,target_h4,target_h5
predispatch_rrp_nsw_h3,4,1,2,383,172
predispatch_rrp_nsw_h4,12,6,1,2,342
predispatch_rrp_nsw_h12,98,146,113,204,111


arcsinh


,target_h1,target_h2,target_h3,target_h4,target_h5
qld_price_asinh_rmean_2016,640,447,480,187,496
nsw_price_rmean_24,53,419,723,857,379
nsw_price_rmean_48,666,702,503,426,730


spikes


,target_h1,target_h2,target_h3,target_h4,target_h5
nsw_price_rmean_24,205,468,480,936,1773
predispatch_rrp_nsw_h21,845,348,1155,1625,796
nsw_price_rmean_48,160,235,268,89,317


dips


,target_h1,target_h2,target_h3,target_h4,target_h5
predispatch_totaldemand_nsw_h51,1518,1596,1616,958,1620
pd_demand_qld_fmax_24h,117,530,1137,1567,1620
pdpasa_maxsparecapacity_nsw_h5,1648,690,385,544,633


In [8]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 35 variable(s); kernel rss 0.87G, 10.3G RAM free now
